# ECCRE | 03: Macro Stress-Testing & Rating Migration Analytics

Executes macro scenario shifts across wholesale energy sectors and measures capital depletion (Delta EL).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.risk.rating_engine import assign_rating_bands
from src.risk.el_calculator import compute_expected_loss
from src.risk.limit_monitor import evaluate_portfolio_limits
from src.risk.stress_tester import run_portfolio_stress_test, run_counterparty_stress_simulation, compute_rating_migration_matrix

%matplotlib inline


### 1. Load Scored Portfolio and Compute Baseline Expected Loss


In [ ]:
df_scored = pd.read_parquet("data/processed/model_predictions.parquet")
print(f"Total Counterparties: {len(df_scored)}")
print(f"Total Baseline EAD: ${df_scored['ead'].sum():,.2f}M")
print(f"Total Baseline EL:  ${df_scored['expected_loss'].sum():,.2f}M")


### 2. Multi-Scenario Portfolio Stress Loss Deltas


In [ ]:
stress_summary = run_portfolio_stress_test(df_scored, "config/stress_scenarios.yaml")
stress_summary


### 3. Rating Migration Matrix under Oil Crash (-50%)


In [ ]:
stressed_cp = run_counterparty_stress_simulation(
    df_scored,
    scenario_key="oil_crash_2008_style",
    scenario_config_path="config/stress_scenarios.yaml",
    rating_config_path="config/rating_bands.yaml"
)

migration_matrix = compute_rating_migration_matrix(stressed_cp)

plt.figure(figsize=(9, 7))
sns.heatmap(migration_matrix, annot=True, fmt="d", cmap="Blues", cbar=True)
plt.title("Credit Rating Migration Matrix: Baseline vs. Oil Crash Scenario")
plt.xlabel("Stressed Rating Band")
plt.ylabel("Baseline Rating Band")
plt.tight_layout()
plt.show()
